# rebound_state_factor_bc_replay_v1

Notebook-first BC-only replay for the rebound-state factor target contract. This diagnostic checks whether the revised target contract is learnable before any PPO/checkpoint training is allowed.

Governance fields:

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1` / current model-retention gate.
- **Direct comparison baseline**: `rebound_state_factor_target_contract_v1` plus the older `sparse_release_bc_ppo_screen_v1` handoff.
- **Reference buy/sell baseline**: `soxlbacktest_qlib_walkforward_baseline_v1`, candidate `wf_pass_ytd18_15_15_02`.
- **Role**: `diagnose`.
- **Changed layer**: `model_selection` / `target_contract` / BC warm-start only; no retained actor is produced.
- **Target windows**: `wf_2020`, `wf_2022`, `wf_2023_2024`.
- **Acceptance criteria**: full readiness requires all required walk-forward windows to have non-leaking target rows, BC test MaxDD `< 20%`, relative return `>= 30pp`, and sparse-slice accuracy `>= 0.60` on release and factor rebound slices.
- **Failure mode**: missing non-leaking target rows, failed sparse slice fit, drawdown over 20%, or weak relative return.

This card is intentionally diagnostic. Passing evaluated folds is not enough for promotion unless `wf_2020` target coverage, fixed-split replay, and full walk-forward/YTD gates are also proven.


In [1]:
from __future__ import annotations

import json
import math
import random
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.evaluation import evaluate_split
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'rebound_state_factor_bc_replay_v1'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
SOURCE_EXPERIMENT = 'rebound_state_factor_target_contract_v1'
REFERENCE_EXPERIMENT = 'soxlbacktest_qlib_walkforward_baseline_v1'
REFERENCE_CANDIDATE = 'wf_pass_ytd18_15_15_02'
CARD = ROOT / 'runtime/model_selection/card' / EXPERIMENT
SOURCE_CARD = ROOT / 'runtime/model_selection/card' / SOURCE_EXPERIMENT
REFERENCE_CARD = ROOT / 'runtime/model_selection/card' / REFERENCE_EXPERIMENT
TARGETS_PATH = SOURCE_CARD / 'selected_target_contract.csv'
SOURCE_SUMMARY_PATH = SOURCE_CARD / 'summary.json'
REFERENCE_METRICS_PATH = REFERENCE_CARD / 'candidate_metrics.csv'
CARD.mkdir(parents=True, exist_ok=True)

SEED = 20260614
TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 20.0
SLICE_ACCURACY_MIN = 0.60
SLICE_SAMPLES_MIN = 5
BC_TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=300, lr=1e-3, batch_size=128, seed=SEED)
BC_WEIGHT_CONFIG = StateObjectiveBCSampleWeightConfig(
    rebound_multiplier=4.0,
    downtrend_multiplier=2.0,
    release_gate_multiplier=12.0,
    objective_weight_power=0.0,
    class_balanced_beta=0.0,
    weight_cap=30.0,
)

WALK_FORWARD_WINDOWS = {
    'wf_2020': {
        'target_fold': 'wf_2020_train',
        'train': ('2010-03-11', '2018-12-31'),
        'valid': ('2019-01-01', '2019-12-31'),
        'test': ('2020-01-01', '2020-12-31'),
    },
    'wf_2022': {
        'target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
    },
    'wf_2023_2024': {
        'target_fold': 'wf_2023_2024_train',
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
    },
}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.floating, float)):
        v = float(value)
        return None if not math.isfinite(v) else v
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')


def load_targets(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    frame['datetime'] = pd.to_datetime(frame['datetime']).dt.normalize()
    for col in ('is_rebound_cluster', 'is_downtrend_state', 'release_gate_state', 'factor_rebound_state'):
        if col in frame.columns:
            if frame[col].dtype == object:
                frame[col] = frame[col].astype(str).str.lower().isin({'true', '1', 'yes'})
            else:
                frame[col] = frame[col].fillna(False).astype(bool)
    return frame


def load_reference_metrics(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    frame = frame[(frame['candidate'] == REFERENCE_CANDIDATE) & (frame['split_type'] == 'walk_forward_test')].copy()
    frame['reference_relative_return_pp'] = frame['rulebase_return_pct'] - frame['soxl_buyhold_return_pct']
    frame['reference_avg_position'] = frame['average_position_pct'] / 100.0
    return frame


def reference_row(reference_metrics: pd.DataFrame, fold: str) -> dict[str, float | str | None]:
    rows = reference_metrics[reference_metrics['window'] == fold]
    if rows.empty:
        return {
            'reference_candidate': REFERENCE_CANDIDATE,
            'reference_return_pct': None,
            'reference_relative_return_pp': None,
            'reference_max_dd_pct': None,
            'reference_avg_position': None,
        }
    row = rows.iloc[0]
    return {
        'reference_candidate': str(row['candidate']),
        'reference_return_pct': float(row['rulebase_return_pct']),
        'reference_relative_return_pp': float(row['reference_relative_return_pp']),
        'reference_max_dd_pct': float(row['rulebase_max_drawdown_pct']),
        'reference_avg_position': float(row['reference_avg_position']),
        'reference_trade_count': int(row['trade_count']),
    }


def evaluate_actor(actor, frame: pd.DataFrame, profile) -> dict[str, float]:
    delta_levels = None
    if bool(profile.model.use_hybrid_overlay):
        delta_levels = np.asarray(profile.model.delta_levels, dtype=np.float32)
    return evaluate_split(
        actor,
        frame,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        action_levels=action_levels_for_profile(profile),
        observation_window=int(profile.model.observation_window),
        hard_floor=profile.model.hard_floor,
        use_hybrid_overlay=bool(profile.model.use_hybrid_overlay),
        delta_levels=delta_levels,
        feature_columns=feature_columns_for_packs(profile.feature.packs, profile_name=profile.name),
        mode_router_config=profile.mode_router if bool(profile.mode_router.enabled) else None,
        position_mapping=profile.model.position_mapping,
    )


def slice_gate(slice_metrics: pd.DataFrame) -> tuple[bool, dict[str, object]]:
    details: dict[str, object] = {}
    required = ['all', 'release_gate_state', 'factor_rebound_state']
    by_slice = slice_metrics.set_index('slice') if not slice_metrics.empty else pd.DataFrame()
    failed: list[str] = []
    for name in required:
        if name not in by_slice.index:
            details[name] = {'samples': 0, 'accuracy': None, 'passed': False}
            failed.append(name)
            continue
        row = by_slice.loc[name]
        samples = int(row['samples'])
        accuracy = float(row['accuracy']) if pd.notna(row['accuracy']) else float('nan')
        min_samples = 1 if name == 'all' else SLICE_SAMPLES_MIN
        passed = bool(samples >= min_samples and math.isfinite(accuracy) and accuracy >= SLICE_ACCURACY_MIN)
        details[name] = {'samples': samples, 'accuracy': accuracy, 'min_samples': min_samples, 'passed': passed}
        if not passed:
            failed.append(name)
    return len(failed) == 0, {'required_slices': required, 'failed_slices': failed, 'details': details}


def target_fold_rows(targets: pd.DataFrame, fold: str, window: dict) -> tuple[pd.DataFrame, dict[str, int]]:
    train_start, train_end = pd.Timestamp(window['train'][0]), pd.Timestamp(window['train'][1])
    valid_start, valid_end = pd.Timestamp(window['valid'][0]), pd.Timestamp(window['valid'][1])
    test_start, test_end = pd.Timestamp(window['test'][0]), pd.Timestamp(window['test'][1])
    fold_targets = targets[targets['fold'] == window['target_fold']].copy()
    train_rows = fold_targets[(fold_targets['datetime'] >= train_start) & (fold_targets['datetime'] <= train_end)].copy()
    overlap = {
        'fold_rows': int(len(fold_targets)),
        'train_rows': int(len(train_rows)),
        'valid_overlap_rows': int(((fold_targets['datetime'] >= valid_start) & (fold_targets['datetime'] <= valid_end)).sum()),
        'test_overlap_rows': int(((fold_targets['datetime'] >= test_start) & (fold_targets['datetime'] <= test_end)).sum()),
    }
    return train_rows, overlap


def metric_row(fold: str, split: str, metrics: dict[str, float]) -> dict[str, float | str]:
    return {
        'fold': fold,
        'split': split,
        'return_pct': float(metrics['return']),
        'sharpe': float(metrics['sharpe']),
        'max_dd_pct': float(metrics['max_dd']),
        'avg_pos': float(metrics['avg_pos']),
        'stress_avg_pos': float(metrics['stress_avg_pos']),
        'stress_days': float(metrics['stress_days']),
        'buy_and_hold_return_pct': float(metrics['buy_and_hold_return']),
        'relative_return_pct': float(metrics['relative_return']),
    }


set_seed(SEED)
profile = load_profile(PROFILE_NAME)
source_summary = json.loads(SOURCE_SUMMARY_PATH.read_text(encoding='utf-8'))
targets = load_targets(TARGETS_PATH)
reference_metrics = load_reference_metrics(REFERENCE_METRICS_PATH)

init_qlib(QLIB_DATA_PATH)

fit_rows: list[dict[str, object]] = []
slice_rows: list[pd.DataFrame] = []
metric_rows: list[dict[str, object]] = []
fold_score_rows: list[dict[str, object]] = []
action_rows: list[pd.DataFrame] = []
baseline_rows: list[dict[str, object]] = []

for fold, window in WALK_FORWARD_WINDOWS.items():
    set_seed(SEED)
    train_targets, overlap = target_fold_rows(targets, fold, window)
    ref = reference_row(reference_metrics, fold)
    base_score = {
        'fold': fold,
        'target_fold': window['target_fold'],
        **overlap,
        **ref,
        'status': 'evaluated',
    }
    if train_targets.empty:
        base_score.update({
            'status': 'skipped_no_nonleaking_targets',
            'fold_gate': False,
            'candidate_gate': False,
            'slice_gate': False,
            'failed_checks': 'missing_nonleaking_target_rows',
        })
        fold_score_rows.append(base_score)
        baseline_rows.append(base_score)
        continue
    if overlap['test_overlap_rows']:
        base_score.update({
            'status': 'blocked_target_leakage',
            'fold_gate': False,
            'candidate_gate': False,
            'slice_gate': False,
            'failed_checks': 'target_overlap_with_valid_or_test',
        })
        fold_score_rows.append(base_score)
        baseline_rows.append(base_score)
        continue

    dataset = build_dataset(
        train_seg=window['train'],
        valid_seg=window['valid'],
        test_seg=window['test'],
        ablate_vix=False,
        feature_packs=profile.feature.packs,
        profile_name=profile.name,
    )
    train_df = segment_frame(dataset, 'train')
    valid_df = segment_frame(dataset, 'valid')
    test_df = segment_frame(dataset, 'test')

    fit = fit_state_objective_bc_actor_from_frame(
        train_df,
        targets=train_targets,
        profile=profile,
        train_config=BC_TRAIN_CONFIG,
        sample_weight_config=BC_WEIGHT_CONFIG,
        rng_seed=SEED,
    )
    fit_row = {'fold': fold, **overlap, **fit.metrics}
    fit_rows.append(fit_row)

    action_dist = fit.action_distribution.copy()
    action_dist.insert(0, 'fold', fold)
    action_rows.append(action_dist)

    # Rebuild metadata through the public dataset helper so slice evidence stays explicit.
    bc_dataset = build_state_objective_bc_dataset_from_frame(
        train_df,
        targets=train_targets,
        profile=profile,
        rng_seed=SEED,
    )
    slices = evaluate_bc_actor_slice_fit(
        fit.actor,
        fit.obs,
        fit.actions,
        bc_dataset.metadata,
        slice_cols=('release_gate_state', 'factor_rebound_state', 'is_rebound_cluster', 'is_downtrend_state'),
    )
    slices.insert(0, 'fold', fold)
    slice_rows.append(slices)
    slice_ok, slice_detail = slice_gate(slices.drop(columns=['fold']))

    valid_metrics = evaluate_actor(fit.actor, valid_df, profile)
    test_metrics = evaluate_actor(fit.actor, test_df, profile)
    metric_rows.append(metric_row(fold, 'valid', valid_metrics))
    metric_rows.append(metric_row(fold, 'walk_forward_test', test_metrics))

    candidate_gate = bool(test_metrics['max_dd'] < TARGET_MAX_DD_PCT and test_metrics['relative_return'] >= TARGET_RELATIVE_RETURN_PP)
    failed_checks: list[str] = []
    if test_metrics['max_dd'] >= TARGET_MAX_DD_PCT:
        failed_checks.append('test_max_dd')
    if test_metrics['relative_return'] < TARGET_RELATIVE_RETURN_PP:
        failed_checks.append('test_relative_return')
    if not slice_ok:
        failed_checks.append('slice_gate')
    fold_gate = bool(candidate_gate and slice_ok)
    model_vs_ref_return_delta = None if ref['reference_return_pct'] is None else float(test_metrics['return'] - ref['reference_return_pct'])
    model_vs_ref_dd_delta = None if ref['reference_max_dd_pct'] is None else float(test_metrics['max_dd'] - ref['reference_max_dd_pct'])
    qlib_reference_drawdown_gate = bool(ref['reference_max_dd_pct'] is not None and ref['reference_max_dd_pct'] < TARGET_MAX_DD_PCT)
    score = {
        **base_score,
        'status': 'evaluated',
        'bc_samples': int(fit.metrics['samples']),
        'bc_after_accuracy': float(fit.metrics['after_accuracy']),
        'valid_return_pct': float(valid_metrics['return']),
        'valid_relative_return_pp': float(valid_metrics['relative_return']),
        'valid_max_dd_pct': float(valid_metrics['max_dd']),
        'test_return_pct': float(test_metrics['return']),
        'test_relative_return_pp': float(test_metrics['relative_return']),
        'test_max_dd_pct': float(test_metrics['max_dd']),
        'test_avg_pos': float(test_metrics['avg_pos']),
        'slice_gate': slice_ok,
        'slice_gate_detail': json.dumps(json_safe(slice_detail), sort_keys=True),
        'candidate_gate': candidate_gate,
        'fold_gate': fold_gate,
        'qlib_reference_drawdown_gate': qlib_reference_drawdown_gate,
        'model_vs_reference_return_delta_pct': model_vs_ref_return_delta,
        'model_vs_reference_max_dd_delta_pct': model_vs_ref_dd_delta,
        'failed_checks': ','.join(failed_checks),
    }
    fold_score_rows.append(score)
    baseline_rows.append(score)

fit_metrics = pd.DataFrame(fit_rows)
slice_metrics = pd.concat(slice_rows, ignore_index=True) if slice_rows else pd.DataFrame()
replay_metrics = pd.DataFrame(metric_rows)
fold_scores = pd.DataFrame(fold_score_rows)
action_distribution = pd.concat(action_rows, ignore_index=True) if action_rows else pd.DataFrame()
baseline_comparison = pd.DataFrame(baseline_rows)

fit_metrics.to_csv(CARD / 'bc_fit_metrics.csv', index=False)
slice_metrics.to_csv(CARD / 'bc_slice_metrics.csv', index=False)
replay_metrics.to_csv(CARD / 'fold_replay_metrics.csv', index=False)
fold_scores.to_csv(CARD / 'fold_gate_scores.csv', index=False)
action_distribution.to_csv(CARD / 'bc_action_distribution.csv', index=False)
baseline_comparison.to_csv(CARD / 'soxlbacktest_qlib_baseline_comparison.csv', index=False)

evaluated = fold_scores[fold_scores['status'] == 'evaluated'].copy()
missing_folds = [fold for fold in WALK_FORWARD_WINDOWS if fold not in set(evaluated['fold'])]
failed_folds = fold_scores.loc[~fold_scores.get('fold_gate', False).fillna(False).astype(bool), 'fold'].tolist()
evaluated_gate = bool(not evaluated.empty and evaluated['fold_gate'].fillna(False).astype(bool).all())
full_required_gate = bool(not missing_folds and evaluated_gate)
if full_required_gate:
    decision = 'rebound_state_factor_bc_replay_full_gate_passed_no_promotion'
    next_mechanism = 'run_full_fixed_split_walkforward_and_ytd_alignment_before_actor_retention'
    blocker = 'full_replay_required_before_promotion'
elif missing_folds and not evaluated_gate:
    decision = 'rebound_state_factor_bc_replay_partial_missing_wf_targets_and_slice_failed_no_training'
    next_mechanism = 'repair_nonleaking_wf2020_target_contract_and_retune_wf2023_release_factor_slice_weighting_before_training'
    blocker = 'missing_nonleaking_wf2020_targets_and_wf2023_slice_fit_failed'
elif missing_folds:
    decision = 'rebound_state_factor_bc_replay_partial_missing_wf_targets_no_training'
    next_mechanism = 'repair_nonleaking_wf2020_target_contract_or_build_pre2020_factor_targets_before_training'
    blocker = 'missing_nonleaking_wf2020_targets'
else:
    decision = 'rebound_state_factor_bc_replay_failed_no_training'
    next_mechanism = 'retune_target_contract_or_bc_slice_weighting_before_ppo'
    blocker = 'bc_candidate_or_slice_gate_failed'

metrics_for_frontier = []
for row in replay_metrics[replay_metrics['split'] == 'walk_forward_test'].to_dict('records'):
    metrics_for_frontier.append({
        'window': row['fold'],
        'split_type': 'walk_forward_test',
        'return_pct': row['return_pct'],
        'max_dd_pct': row['max_dd_pct'],
        'relative_return_pct': row['relative_return_pct'],
        'avg_pos': row['avg_pos'],
    })

summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'profile': PROFILE_NAME,
    'source_card': SOURCE_EXPERIMENT,
    'source_target_contract': SOURCE_EXPERIMENT,
    'source_target_path': str(TARGETS_PATH.relative_to(ROOT)),
    'source_target_decision': source_summary.get('decision'),
    'reference_baseline': f'{REFERENCE_EXPERIMENT}:{REFERENCE_CANDIDATE}',
    'target_contract_gate': bool(source_summary.get('target_contract_gate')),
    'bc_readiness_gate': full_required_gate,
    'bc_evaluated_fold_gate': evaluated_gate,
    'training_allowed': False,
    'promotion_allowed': False,
    'next_training_allowed_after_target_revision': False,
    'next_mechanism': next_mechanism,
    'primary_blocker': blocker,
    'recommended_primary_blocker': blocker,
    'target_relative_return_pp': TARGET_RELATIVE_RETURN_PP,
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'target_relative_return_by_window_pp': {fold: TARGET_RELATIVE_RETURN_PP for fold in WALK_FORWARD_WINDOWS},
    'required_wf_windows': list(WALK_FORWARD_WINDOWS),
    'evaluated_folds': evaluated['fold'].tolist(),
    'missing_folds': missing_folds,
    'failed_folds': failed_folds,
    'fold_gate_pass_count': int(fold_scores.get('fold_gate', pd.Series(dtype=bool)).fillna(False).astype(bool).sum()),
    'candidate_gate_pass_count': int(fold_scores.get('candidate_gate', pd.Series(dtype=bool)).fillna(False).astype(bool).sum()),
    'slice_gate_pass_count': int(fold_scores.get('slice_gate', pd.Series(dtype=bool)).fillna(False).astype(bool).sum()),
    'qlib_reference_drawdown_gate_pass_count': int(fold_scores.get('qlib_reference_drawdown_gate', pd.Series(dtype=bool)).fillna(False).astype(bool).sum()),
    'metrics': metrics_for_frontier,
    'sidecars': {
        'bc_fit_metrics': str((CARD / 'bc_fit_metrics.csv').relative_to(ROOT)),
        'bc_slice_metrics': str((CARD / 'bc_slice_metrics.csv').relative_to(ROOT)),
        'fold_replay_metrics': str((CARD / 'fold_replay_metrics.csv').relative_to(ROOT)),
        'fold_gate_scores': str((CARD / 'fold_gate_scores.csv').relative_to(ROOT)),
        'bc_action_distribution': str((CARD / 'bc_action_distribution.csv').relative_to(ROOT)),
        'baseline_comparison': str((CARD / 'soxlbacktest_qlib_baseline_comparison.csv').relative_to(ROOT)),
    },
}
selection = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'profile': PROFILE_NAME,
    'source_card': SOURCE_EXPERIMENT,
    'reference_baseline': summary['reference_baseline'],
    'bc_readiness_gate': full_required_gate,
    'bc_evaluated_fold_gate': evaluated_gate,
    'training_allowed': False,
    'promotion_allowed': False,
    'next_mechanism': next_mechanism,
    'primary_blocker': blocker,
    'required_wf_windows': list(WALK_FORWARD_WINDOWS),
    'evaluated_folds': evaluated['fold'].tolist(),
    'missing_folds': missing_folds,
    'failed_folds': failed_folds,
}

write_json(CARD / 'summary.json', summary)
write_json(CARD / 'selection.json', selection)

display(Markdown(
    f"## Conclusion\n\n"
    f"- Decision: `{decision}`.\n"
    f"- Evaluated folds: `{evaluated['fold'].tolist()}`; missing folds: `{missing_folds}`.\n"
    f"- Fold gates passed: `{summary['fold_gate_pass_count']}/{len(WALK_FORWARD_WINDOWS)}`; BC readiness gate: `{full_required_gate}`.\n"
    f"- Next: `{next_mechanism}`.\n"
))

summary


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[32271:MainThread](2026-06-14 20:37:49,934) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[32271:MainThread](2026-06-14 20:37:50,357) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[32271:MainThread](2026-06-14 20:37:50,358) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


[32271:MainThread](2026-06-14 20:37:51,426) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


...

[32271:MainThread](2026-06-14 20:37:54,938) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_32271/29523582.py:442: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  'qlib_reference_drawdown_gate_pass_count': int(fold_scores.get('qlib_reference_drawdown_gate', pd.Series(dtype=bool)).fillna(False).astype(bool).sum()),


## Conclusion

- Decision: `rebound_state_factor_bc_replay_partial_missing_wf_targets_and_slice_failed_no_training`.
- Evaluated folds: `['wf_2022', 'wf_2023_2024']`; missing folds: `['wf_2020']`.
- Fold gates passed: `1/3`; BC readiness gate: `False`.
- Next: `repair_nonleaking_wf2020_target_contract_and_retune_wf2023_release_factor_slice_weighting_before_training`.


{'experiment': 'rebound_state_factor_bc_replay_v1',
 'decision': 'rebound_state_factor_bc_replay_partial_missing_wf_targets_and_slice_failed_no_training',
 'generated_at': '2026-06-14T12:37:56.931398+00:00',
 'profile': 'sparse_release_bc_ppo_screen_v1',
 'source_card': 'rebound_state_factor_target_contract_v1',
 'source_target_contract': 'rebound_state_factor_target_contract_v1',
 'source_target_path': 'runtime/model_selection/card/rebound_state_factor_target_contract_v1/selected_target_contract.csv',
 'source_target_decision': 'rebound_state_factor_target_contract_ready_for_label_stability_no_training',
 'reference_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1:wf_pass_ytd18_15_15_02',
 'target_contract_gate': True,
 'bc_readiness_gate': False,
 'bc_evaluated_fold_gate': False,
 'training_allowed': False,
 'promotion_allowed': False,
 'next_training_allowed_after_target_revision': False,
 'next_mechanism': 'repair_nonleaking_wf2020_target_contract_and_retune_wf2023_release_fac